In [5]:
import torch.nn as nn
import torch


class Encoder(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, n_encoder_layers):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, d_ff, n_heads) for _ in range(n_encoder_layers)])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, d_model, d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(q=x))
        x = self.norm2(x + self.ffn(x))
        return x


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(torch.relu(self.linear1(x)))


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, attn_dim: int, output_dim: int, num_heads: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.attn_dim = attn_dim
        self.output_dim = output_dim
        self.num_heads = num_heads
        self.head_dim = attn_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)

        self.out_proj = nn.Linear(self.attn_dim, self.output_dim, bias=False)

    def forward(self, q, k=None, v=None, mask=None):
        if k is None: k = q
        if v is None: v = q

        batch_size, seq_len_q, _ = q.shape
        seq_len_k = k.shape[1]

        q = self.q_proj(q).view(batch_size, seq_len_q, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(k).view(batch_size, seq_len_k, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(v).view(batch_size, v.shape[1], self.num_heads, self.head_dim).transpose(1, 2)

        attn_score = torch.matmul(q, k.transpose(-2, -1))
        attn_score = attn_score / torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32))

        if mask is not None:
            mask = mask.unsqueeze(0).unsqueeze(1)
            attn_score = attn_score.masked_fill(mask == 0, -1e9)

        attn_weight = torch.softmax(attn_score, dim=-1)

        output = torch.matmul(attn_weight, v)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len_q, self.attn_dim)
        return self.out_proj(output)


In [6]:
class Decoder(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, n_decoder_layers):
        super().__init__()
        self.layers = nn.ModuleList([DecoderLayer(d_model, d_ff, n_heads) for _ in range(n_decoder_layers)])

    def forward(self, decoder_inputs, encoder_outputs, mask=None):
        for layer in self.layers:
            decoder_inputs = layer(decoder_inputs, encoder_outputs, mask=mask)
        return decoder_inputs


class DecoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_heads):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, d_model, d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, d_model, d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, decoder_inputs, encoder_outputs, mask=None):
        attn1 = self.self_attn(q=decoder_inputs, mask=mask)
        x = self.norm1(decoder_inputs + attn1)

        attn2 = self.cross_attn(q=x, k=encoder_outputs, v=encoder_outputs)
        x = self.norm2(x + attn2)

        ffn_out = self.ffn(x)
        x = self.norm3(x + ffn_out)
        return x


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(torch.relu(self.linear1(x)))

In [7]:
class Transformer(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, n_encoder_layers, n_decoder_layers):
        super().__init__()
        self.n_vocab = n_vocab
        self.d_model = d_model
        self.d_ff = d_ff
        self.n_heads = n_heads
        self.n_encoder_layers = n_encoder_layers
        self.n_decoder_layers = n_decoder_layers
        self.encoder = Encoder(d_model, d_ff, n_heads, n_encoder_layers)
        self.decoder = Decoder(d_model, d_ff, n_heads, n_decoder_layers)
        self.out = nn.Linear(d_model, n_vocab)

    def forward(self, encoder_inputs, decoder_inputs, mask=None):
        encoder_outputs = self.encoder(encoder_inputs)
        decoder_outputs = self.decoder(decoder_inputs, encoder_outputs, mask=mask)
        return self.out(decoder_outputs)


In [8]:
x1 = torch.randn(1, 10, 512)
x2 = torch.randn(1, 10, 512)
seq_len = x1.shape[1]
mask = torch.tril(torch.ones(seq_len, seq_len))
n_vocab = 1000
model = Transformer(d_model=512, d_ff=2048, n_heads=8, n_encoder_layers=6, n_decoder_layers=6)
y = model(x1, x2, mask=mask)
print(x1)
print(x2)
print(y)


tensor([[[ 0.3075,  0.0744,  1.7816,  ...,  1.0160, -1.2475,  1.9919],
         [-1.9148,  1.7759, -0.4599,  ..., -0.3158, -1.6299, -0.7657],
         [-2.3188, -1.2237,  0.7933,  ...,  1.6188,  0.8670,  0.4395],
         ...,
         [ 2.2143, -0.5786,  0.0854,  ...,  1.2617, -0.6471,  0.1506],
         [ 0.8920,  0.7060,  0.0143,  ..., -1.9742,  0.3649, -1.6939],
         [ 1.1854,  1.3976,  1.0137,  ...,  0.4265,  1.0533, -0.2172]]])
tensor([[[-0.4771, -1.4737,  0.3161,  ...,  1.4343,  0.3299, -1.1245],
         [-1.0994,  0.1719, -0.3620,  ...,  2.0648, -0.0668, -0.8949],
         [ 1.6985,  0.9419, -1.9220,  ...,  0.3449,  1.0699, -0.7959],
         ...,
         [-0.4364,  1.7919, -0.3834,  ..., -0.3374,  0.4437,  0.5851],
         [-1.0036, -0.4384, -1.1732,  ..., -1.1695,  0.9617, -0.5486],
         [-0.4075,  1.7384, -0.5090,  ..., -1.0828,  0.1406, -0.3968]]])
tensor([[[ 0.0232,  0.4294,  0.0758,  ...,  0.8954, -0.9537, -0.5097],
         [ 0.4072,  0.6945, -0.5652,  ..., -0